# Exploração da Camada Silver — Health Insurance Marketplace

## Propósito

Este notebook documenta a **camada Silver** do Data Lake — os dados após limpeza, tipagem e deduplicação aplicadas pelo job `bronze_to_silver`.

> **Este notebook é o insumo para o desenvolvimento da camada Gold.** Cada seção termina com notas sobre o que essa tabela oferece para construção de agregados e análises.

## O que foi feito na Silver

| Transformação | Detalhes |
|---|---|
| Tipagem numérica | `CopayInnTier1`, `IndividualRate` e ~70 colunas convertidas de string para `double` |
| Tipagem booleana | `IsCovered`, `CoverEntireState`, `IsNewPlan` e ~20 colunas para `boolean` |
| Tipagem temporal | `ImportDate`, `RateEffectiveDate`, `RateExpirationDate` para `timestamp` |
| Tipagem inteira | `BusinessYear`, `Age`, `LimitQty` e ~15 colunas para `integer` |
| Normalização de nulos semânticos | `"No Charge"` → `0.0`; `"Not Applicable"` → `null` |
| Deduplicação | Por `PlanId` + ano, mantendo maior `VersionNum` |
| Tabelas excluídas | Tabelas com prefixo `raw_` (dados brutos anuais) não são promovidas à Silver |

## Tabelas disponíveis na Silver

| Tabela | Linhas | Colunas | Propósito |
|---|---|---|---|
| `rate` | ~12.7 M | 26 | Prêmios mensais por plano/idade/área |
| `plan_attributes` | ~77 K | 178 | Catálogo mestre dos planos |
| `benefits_cost_sharing` | ~5 M | 34 | Benefícios e regras de custo por serviço |
| `service_area` | ~42 K | 20 | Cobertura geográfica das seguradoras |
| `business_rules` | ~21 K | 25 | Regras de elegibilidade por plano |
| `crosswalk2015` | ~132 K | 23 | Linhagem dos planos 2014 → 2015 |
| `crosswalk2016` | ~150 K | 23 | Linhagem dos planos 2015 → 2016 |
| `network` | ~3.8 K | 16 | Metadados das redes de prestadores |

## Como executar

1. Dev Container ativo com credenciais AWS atualizadas.
2. Kernel **Python 3** (`/opt/conda/bin/python`).
3. Execute as células sequencialmente.

> Usa `awswrangler` para consultar o Athena — não requer PySpark.

---
## 1. Configuração

In [1]:
import awswrangler as wr
import boto3
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 300)
pd.set_option("display.float_format", "{:.4f}".format)

In [2]:
account_id       = boto3.client("sts").get_caller_identity()["Account"]
SILVER_DATABASE  = "eedb015_silver"
S3_ATHENA_OUTPUT = f"s3://{account_id}-eedb015-athena-results/"

print(f"Account ID       : {account_id}")
print(f"Silver Database  : {SILVER_DATABASE}")
print(f"Athena Output    : {S3_ATHENA_OUTPUT}")

Account ID       : 637423524537
Silver Database  : eedb015_silver
Athena Output    : s3://637423524537-eedb015-athena-results/


---
## 2. Funções Auxiliares

In [3]:
def query(sql: str, **kwargs) -> pd.DataFrame:
    """Executa SQL no Athena via awswrangler."""
    return wr.athena.read_sql_query(
        sql=sql,
        database=SILVER_DATABASE,
        s3_output=S3_ATHENA_OUTPUT,
        ctas_approach=False,
        **kwargs,
    )


def get_schema(table_name: str) -> pd.DataFrame:
    """Retorna schema da tabela (colunas e tipos) do Glue Catalog."""
    df = wr.catalog.table(database=SILVER_DATABASE, table=table_name)
    return df[["Column Name", "Type", "Partition"]].copy()


def sample_table(table_name: str, n: int = 10) -> pd.DataFrame:
    """Retorna n linhas de amostra via Athena."""
    return query(f'SELECT * FROM "{SILVER_DATABASE}"."{table_name}" LIMIT {n}')


def null_profile(table_name: str, cols: list[str]) -> pd.DataFrame:
    """Calcula contagem e % de nulos para as colunas informadas."""
    null_exprs = ",\n    ".join([
        f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}"
        for c in cols
    ])
    sql = f"""
        SELECT
            COUNT(*) AS total_rows,
            {null_exprs}
        FROM "{SILVER_DATABASE}"."{table_name}"
    """
    raw = query(sql)
    total = raw["total_rows"].iloc[0]
    records = [{"column": c, "null_count": int(raw[c].iloc[0]),
                "null_pct": round(100.0 * int(raw[c].iloc[0]) / total, 2)}
               for c in cols]
    return pd.DataFrame(records).sort_values("null_pct", ascending=False)


def numeric_stats(table_name: str, cols: list[str]) -> pd.DataFrame:
    """Calcula min, max, média e desvio-padrão para colunas numéricas."""
    exprs = ",\n    ".join([
        f"MIN({c}) AS {c}_min, MAX({c}) AS {c}_max, "
        f"ROUND(AVG({c}), 4) AS {c}_avg"
        for c in cols
    ])
    raw = query(f'SELECT {exprs} FROM "{SILVER_DATABASE}"."{table_name}"')
    rows = []
    for c in cols:
        rows.append({"column": c,
                     "min": raw[f"{c}_min"].iloc[0],
                     "max": raw[f"{c}_max"].iloc[0],
                     "avg": raw[f"{c}_avg"].iloc[0]})
    return pd.DataFrame(rows)


def categorical_dist(table_name: str, col: str, limit: int = 15) -> pd.DataFrame:
    """Retorna distribuição de frequência de uma coluna categórica."""
    return query(f"""
        SELECT
            {col} AS value,
            COUNT(*) AS n,
            ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
        FROM "{SILVER_DATABASE}"."{table_name}"
        GROUP BY {col}
        ORDER BY n DESC
        LIMIT {limit}
    """)


print("Funções auxiliares carregadas.")

Funções auxiliares carregadas.


---
## 3. Descoberta das Tabelas Silver

In [4]:
tables_df = wr.catalog.tables(database=SILVER_DATABASE)
print(f"Total de tabelas no database '{SILVER_DATABASE}': {len(tables_df)}")
display(tables_df[["Table", "TableType"]].sort_values("Table").reset_index(drop=True))

Total de tabelas no database 'eedb015_silver': 8


,Table,TableType
0,benefits_cost_sharing,EXTERNAL_TABLE
1,business_rules,EXTERNAL_TABLE
2,crosswalk2015,EXTERNAL_TABLE
3,crosswalk2016,EXTERNAL_TABLE
4,network,EXTERNAL_TABLE
5,plan_attributes,EXTERNAL_TABLE
6,rate,EXTERNAL_TABLE
7,service_area,EXTERNAL_TABLE


---
## ⚠️ Atenção: Formatos de PlanId são Incompatíveis entre Tabelas

Este é o ponto de atenção **mais crítico** para o desenvolvimento da Gold.
O campo `PlanId` **não tem o mesmo formato** em todas as tabelas Silver:

| Tabela | Formato de PlanId | Exemplo | Tamanho |
|---|---|---|---|
| `rate` | Sem sufixo de variante | `21989AK0010001` | 14 chars |
| `plan_attributes` | Com sufixo `-XX` (variante) | `21989AK0020002-00` | 17 chars |
| `benefits_cost_sharing` | Com sufixo `-XX` (variante) | `21989AK0020002-00` | 17 chars |
| `business_rules` | Com sufixo `-XX` (variante) | `21989AK0020002-00` | 17 chars |
| `crosswalk2015/2016` | Sem sufixo (base 14 chars) | `21989AK0010001` | 14 chars |

### Estrutura do HIOS Plan ID

```
Formato completo (plan_attributes, benefits_cost_sharing, business_rules):
  21989  AK  001  0001  -00
  IIIII  SS  PPP  VVVV  -VR
  │      │   │    │     └─ VariantId (sufixo: 00=base, 73=CSR 94%, 87=CSR 87%, etc.)
  │      │   │    └─────── PlanId (4 chars)
  │      │   └──────────── ProductId (3 chars)
  │      └──────────────── StateCode (2 chars)
  └─────────────────────── IssuerId (5 chars)

Formato rate / crosswalk (14 chars, sem sufixo):
  21989AK0010001  →  IssuerId(5) + StateCode(2) + ProductId(3) + PlanId(4)
```

### Sintaxe correta de JOIN no Athena

```sql
-- ✅ rate → plan_attributes (remover sufixo de plan_attributes)
JOIN plan_attributes pa
  ON SUBSTR(pa.PlanId, 1, 14) = r.PlanId
 AND pa.BusinessYear = r.BusinessYear

-- ✅ rate → benefits_cost_sharing (mesma lógica)
JOIN benefits_cost_sharing bcs
  ON SUBSTR(bcs.PlanId, 1, 14) = r.PlanId
 AND bcs.BusinessYear = r.BusinessYear

-- ✅ plan_attributes → benefits_cost_sharing (ambos têm sufixo — join direto)
JOIN benefits_cost_sharing bcs
  ON bcs.PlanId = pa.PlanId
 AND bcs.BusinessYear = pa.BusinessYear

-- ✅ plan_attributes → business_rules (ambos têm sufixo — join direto)
JOIN business_rules br
  ON br.PlanId = pa.PlanId
 AND br.BusinessYear = pa.BusinessYear

-- ✅ crosswalk → rate (ambos sem sufixo — join direto)
JOIN rate r
  ON r.PlanId = c.CrosswalkPlanId

-- ✅ crosswalk → plan_attributes (crosswalk sem sufixo, plan_attributes com)
JOIN plan_attributes pa
  ON SUBSTR(pa.PlanId, 1, 14) = c.PlanId
 AND pa.BusinessYear = c.BusinessYear
```

### Filtro para excluir variantes CSR (manter apenas plano base)

```sql
-- Na tabela plan_attributes: manter só o plano base (sufixo '00')
WHERE RIGHT(PlanId, 2) = '00'
-- equivalente a: SUBSTR(PlanId, 16, 2) = '00'
```

---
## 4. Tabela: `rate`

**Descrição:** Prêmios mensais por plano, segmentados por faixa etária, perfil de tabaco e área geográfica.

**Dimensão:** ~12,7 milhões de linhas · 26 colunas

**Transformações Silver aplicadas:**
- `IndividualRate`, `IndividualTobaccoRate`, `Couple` e variantes de família → `double`
- `RateEffectiveDate`, `RateExpirationDate`, `ImportDate` → `timestamp`
- `BusinessYear`, `Age` → `integer`

**Colunas relevantes para Gold:**
- `PlanId` — join com `plan_attributes` e `benefits_cost_sharing`
- `BusinessYear` — dimensão temporal para análises 2014-2016
- `StateCode` — dimensão geográfica (competição por estado)
- `Age` — segmentação demográfica
- `Tobacco` — segmentação por perfil de risco
- `IndividualRate` — métrica principal de preço

In [5]:
display(Markdown("### Schema"))
display(get_schema("rate"))

### Schema

,Column Name,Type,Partition
0,businessyear,int,False
1,statecode,string,False
2,issuerid,string,False
3,sourcename,string,False
4,versionnum,string,False
5,importdate,timestamp,False
6,issuerid2,string,False
7,federaltin,string,False
8,rateeffectivedate,timestamp,False
9,rateexpirationdate,timestamp,False


In [6]:
display(Markdown("### Amostra (10 linhas)"))
display(sample_table("rate"))

### Amostra (10 linhas)

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,federaltin,rateeffectivedate,rateexpirationdate,planid,ratingareaid,tobacco,age,individualrate,individualtobaccorate,couple,primarysubscriberandonedependent,primarysubscriberandtwodependents,primarysubscriberandthreeormoredependents,coupleandonedependent,coupleandtwodependents,coupleandthreeormoredependents,rownumber,landinzone_path,ingestion_datetime
0,2014,AK,21989,HIOS,6,NaT,21989,93-0438772,NaT,NaT,21989AK0010001,Rating Area 1,No Preference,<NA>,29.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
1,2014,AK,21989,HIOS,6,NaT,21989,93-0438772,NaT,NaT,21989AK0020001,Rating Area 1,No Preference,<NA>,36.9500,NaN,73.9000,107.6100,107.6100,107.6100,144.5600,144.5600,144.5600,14,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
2,2014,AK,21989,HIOS,6,NaT,21989,93-0438772,NaT,NaT,21989AK0020001,Rating Area 2,No Preference,<NA>,36.9500,NaN,73.9000,107.6100,107.6100,107.6100,144.5600,144.5600,144.5600,15,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
3,2014,AK,21989,HIOS,6,NaT,21989,93-0438772,NaT,NaT,21989AK0010001,Rating Area 1,No Preference,21,32.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
4,2014,AK,21989,HIOS,6,NaT,21989,93-0438772,NaT,NaT,21989AK0010001,Rating Area 1,No Preference,22,32.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
5,2014,AK,21989,HIOS,6,NaT,21989,93-0438772,NaT,NaT,21989AK0020001,Rating Area 3,No Preference,<NA>,36.9500,NaN,73.9000,107.6100,107.6100,107.6100,144.5600,144.5600,144.5600,16,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
6,2014,AK,21989,HIOS,6,NaT,21989,93-0438772,NaT,NaT,21989AK0020002,Rating Area 1,No Preference,<NA>,32.4500,NaN,64.9000,94.5000,94.5000,94.5000,126.9500,126.9500,126.9500,17,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
7,2014,AK,21989,HIOS,6,NaT,21989,93-0438772,NaT,NaT,21989AK0010001,Rating Area 1,No Preference,23,32.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
8,2014,AK,21989,HIOS,6,NaT,21989,93-0438772,NaT,NaT,21989AK0010001,Rating Area 1,No Preference,24,32.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
9,2014,AK,21989,HIOS,6,NaT,21989,93-0438772,NaT,NaT,21989AK0020002,Rating Area 2,No Preference,<NA>,32.4500,NaN,64.9000,94.5000,94.5000,94.5000,126.9500,126.9500,126.9500,18,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00


In [7]:
display(Markdown("### Perfil de Nulos — colunas de preço"))
display(null_profile("rate", [
    "IndividualRate", "IndividualTobaccoRate", "Couple",
    "PrimarySubscriberAndOneDependent", "PrimarySubscriberAndTwoDependents",
    "PrimarySubscriberAndThreeOrMoreDependents",
    "CoupleAndOneDependent", "CoupleAndTwoDependents", "CoupleAndThreeOrMoreDependents",
    "RatingAreaId", "Age",
]))

### Perfil de Nulos — colunas de preço

,column,null_count,null_pct
2,Couple,12653504,99.6800
4,PrimarySubscriberAndTwoDependents,12653504,99.6800
3,PrimarySubscriberAndOneDependent,12653504,99.6800
7,CoupleAndTwoDependents,12653504,99.6800
8,CoupleAndThreeOrMoreDependents,12653504,99.6800
5,PrimarySubscriberAndThreeOrMoreDependents,12653504,99.6800
6,CoupleAndOneDependent,12653504,99.6800
1,IndividualTobaccoRate,7762096,61.1500
10,Age,591497,4.6600
0,IndividualRate,0,0.0000


In [8]:
display(Markdown("### Estatísticas de IndividualRate"))
display(numeric_stats("rate", ["IndividualRate", "IndividualTobaccoRate"]))

### Estatísticas de IndividualRate

,column,min,max,avg
0,IndividualRate,0.0000,999999.0000,4098.0265
1,IndividualTobaccoRate,41.7300,6604.6100,543.6911


In [9]:
display(Markdown("### Distribuição por BusinessYear"))
display(query("""
    SELECT
        BusinessYear,
        COUNT(*)                              AS total_rows,
        COUNT(DISTINCT PlanId)                AS distinct_plans,
        COUNT(DISTINCT StateCode)             AS distinct_states,
        COUNT(DISTINCT IssuerId)              AS distinct_issuers,
        ROUND(MIN(IndividualRate), 2)         AS min_rate,
        ROUND(MAX(IndividualRate), 2)         AS max_rate,
        ROUND(AVG(IndividualRate), 2)         AS avg_rate
    FROM "eedb015_silver"."rate"
    GROUP BY BusinessYear
    ORDER BY BusinessYear
"""))

### Distribuição por BusinessYear

,BusinessYear,total_rows,distinct_plans,distinct_states,distinct_issuers,min_rate,max_rate,avg_rate
0,2014,3796388,6633,36,450,0.0000,999999.0000,12922.2600
1,2015,4676092,10095,37,783,0.0000,9999.9900,329.1600
2,2016,4221965,8887,38,770,0.0000,9999.0000,337.5400


In [10]:
display(Markdown("### Distribuição: Tobacco"))
display(categorical_dist("rate", "Tobacco"))

display(Markdown("### Distribuição: Age (top 20 faixas)"))
display(query("""
    SELECT Age, COUNT(*) AS n
    FROM "eedb015_silver"."rate"
    GROUP BY Age ORDER BY Age
    LIMIT 20
"""))

### Distribuição: Tobacco

,value,n,pct
0,No Preference,7804323,61.4800
1,Tobacco User/Non-Tobacco User,4890122,38.5200


### Distribuição: Age (top 20 faixas)

,Age,n
0,21,275067
1,22,275067
2,23,275067
3,24,275067
4,25,275067
5,26,275067
6,27,275067
7,28,275067
8,29,275067
9,30,275067


In [11]:
display(Markdown("""
### Notas para Gold

- **Q2 / Q5 (Competição × Preço):** agregar `AVG(IndividualRate)` por `StateCode` + `BusinessYear`;
  contar `COUNT(DISTINCT IssuerId)` como proxy de competição.
- **Q4 (Rede × Preço):** fazer join com `plan_attributes` via `PlanId` para trazer `NetworkId`,
  depois agregar preço médio por rede.
- **Q1 (Oncologia):** esta tabela não tem info de benefícios — o join com `benefits_cost_sharing`
  é necessário para ligar preço ao tipo de cobertura.
- **Filtro recomendado:** `WHERE IndividualRate > 0 AND IndividualRate < 3000`
  para excluir outliers (valores zero = planos sem taxa, >3000 = erros de digitação).
- **Atenção:** `Age = 'Family Option'` é um valor especial (não numérico) — tratar separadamente.
"""))


### Notas para Gold

- **Q2 / Q5 (Competição × Preço):** agregar `AVG(IndividualRate)` por `StateCode` + `BusinessYear`;
  contar `COUNT(DISTINCT IssuerId)` como proxy de competição.
- **Q4 (Rede × Preço):** fazer join com `plan_attributes` via `PlanId` para trazer `NetworkId`,
  depois agregar preço médio por rede.
- **Q1 (Oncologia):** esta tabela não tem info de benefícios — o join com `benefits_cost_sharing`
  é necessário para ligar preço ao tipo de cobertura.
- **Filtro recomendado:** `WHERE IndividualRate > 0 AND IndividualRate < 3000`
  para excluir outliers (valores zero = planos sem taxa, >3000 = erros de digitação).
- **Atenção:** `Age = 'Family Option'` é um valor especial (não numérico) — tratar separadamente.


---
## 5. Tabela: `plan_attributes`

**Descrição:** Catálogo mestre dos planos — metadados de cada plano (nível metálico, tipo, deductibles, MOOP, etc.)

**Dimensão:** ~77.000 linhas · 178 colunas

**Transformações Silver aplicadas:**
- ~70 colunas de MOOP/Deductible → `double`
- Booleanos: `IsNewPlan`, `WellnessProgramOffered`, `NationalNetwork`, `DentalOnlyPlan`, etc. → `boolean`
- Inteiros: `BusinessYear`, `OutOfCountryCoverage`, `MultipleInNetworkTiers`, etc. → `integer`

**Colunas relevantes para Gold:**
- `PlanId` — chave primária, join com `rate` e `benefits_cost_sharing`
- `MetalLevel` — segmentação principal (Bronze/Silver/Gold/Platinum/Catastrophic)
- `PlanType` — HMO, PPO, EPO, POS
- `NetworkId` — join com `network`
- `ServiceAreaId` — join com `service_area`
- `IssuerId` — identificador da seguradora (competição)
- `BusinessYear` — dimensão temporal
- Colunas MOOP (`MEHBInnTier1IndividualMOOP`, etc.) — exposição financeira máxima

In [12]:
display(Markdown("### Schema (primeiras 40 colunas — tabela tem 178 colunas no total)"))
schema = get_schema("plan_attributes")
print(f"Total de colunas: {len(schema)}")
display(schema.head(40))

### Schema (primeiras 40 colunas — tabela tem 178 colunas no total)

Total de colunas: 178


,Column Name,Type,Partition
0,avcalculatoroutputnumber,string,False
1,beginprimarycarecostsharingafternumberofvisits,int,False
2,beginprimarycaredeductiblecoinsuranceafternumberofcopays,int,False
3,benefitpackageid,int,False
4,businessyear,int,False
5,csrvariationtype,string,False
6,childonlyoffering,string,False
7,childonlyplanid,string,False
8,compositeratingoffered,string,False
9,dehbcombinnoonfamilymoop,double,False


In [13]:
display(Markdown("### Amostra — colunas principais (10 linhas)"))
display(query("""
    SELECT
        BusinessYear, StateCode, IssuerId, PlanId,
        PlanMarketingName, MetalLevel, PlanType,
        NetworkId, ServiceAreaId,
        IsNewPlan, WellnessProgramOffered, NationalNetwork,
        EHBPercentTotalPremium,
        MEHBInnTier1IndividualMOOP
    FROM "eedb015_silver"."plan_attributes"
    LIMIT 10
"""))

### Amostra — colunas principais (10 linhas)

,BusinessYear,StateCode,IssuerId,PlanId,PlanMarketingName,MetalLevel,PlanType,NetworkId,ServiceAreaId,IsNewPlan,WellnessProgramOffered,NationalNetwork,EHBPercentTotalPremium,MEHBInnTier1IndividualMOOP
0,2014,AK,21989,21989AK0020002-00,Premier,Low,PPO,AKN001,AKS001,True,<NA>,<NA>,NaN,NaN
1,2014,AK,21989,21989AK0020002-01,Premier,Low,PPO,AKN001,AKS001,True,<NA>,<NA>,NaN,NaN
2,2014,AK,21989,21989AK0020001-00,Premier,High,PPO,AKN001,AKS001,True,<NA>,<NA>,NaN,NaN
3,2014,AK,21989,21989AK0010001-00,Premier,Low,PPO,AKN001,AKS001,True,<NA>,<NA>,NaN,NaN
4,2014,AK,21989,21989AK0010001-01,Premier,Low,PPO,AKN001,AKS001,True,<NA>,<NA>,NaN,NaN
5,2014,AK,21989,21989AK0020001-01,Premier,High,PPO,AKN001,AKS001,True,<NA>,<NA>,NaN,NaN
6,2014,AK,73836,73836AK0650002-00,Be Connected,Bronze,PPO,AKN001,AKS001,True,True,<NA>,NaN,NaN
7,2014,AK,73836,73836AK0680004-00,PPO 1000A,Gold,PPO,AKN001,AKS001,True,True,<NA>,NaN,NaN
8,2014,AK,73836,73836AK0680004-01,PPO 1000A,Gold,PPO,AKN001,AKS001,True,True,<NA>,NaN,NaN
9,2014,AK,73836,73836AK0650002-01,Be Connected,Bronze,PPO,AKN001,AKS001,True,True,<NA>,NaN,NaN


In [14]:
display(Markdown("### Distribuição: MetalLevel × PlanType × BusinessYear"))
display(query("""
    SELECT
        BusinessYear,
        MetalLevel,
        PlanType,
        COUNT(*)                   AS plan_count,
        COUNT(DISTINCT IssuerId)   AS distinct_issuers,
        COUNT(DISTINCT StateCode)  AS distinct_states
    FROM "eedb015_silver"."plan_attributes"
    GROUP BY BusinessYear, MetalLevel, PlanType
    ORDER BY BusinessYear, plan_count DESC
"""))

### Distribuição: MetalLevel × PlanType × BusinessYear

,BusinessYear,MetalLevel,PlanType,plan_count,distinct_issuers,distinct_states
0,2014,Silver,HMO,3118,97,26
1,2014,Silver,PPO,2607,89,33
2,2014,Low,PPO,1590,248,36
3,2014,Bronze,HMO,1559,94,27
4,2014,Gold,HMO,1486,97,26
...,...,...,...,...,...,...
96,2016,Silver,Indemnity,4,1,1
97,2016,Gold,Indemnity,4,1,1
98,2016,Platinum,Indemnity,4,1,1
99,2016,High,POS,3,3,3


In [15]:
display(Markdown("### Estatísticas das colunas MOOP e Deductible"))
display(numeric_stats("plan_attributes", [
    "MEHBInnTier1IndividualMOOP",
    "MEHBInnTier1FamilyMOOP",
    "MEHBOutOfNetIndividualMOOP",
    "MEHBDedInnTier1Individual",
    "MEHBDedInnTier1Family",
    "EHBPercentTotalPremium",
]))

display(Markdown("### Nulos nas colunas financeiras chave"))
display(null_profile("plan_attributes", [
    "MetalLevel", "PlanType", "NetworkId", "ServiceAreaId",
    "MEHBInnTier1IndividualMOOP",
    "MEHBDedInnTier1Individual",
    "EHBPercentTotalPremium", "IsNewPlan", "WellnessProgramOffered",
]))

### Estatísticas das colunas MOOP e Deductible

,column,min,max,avg
0,MEHBInnTier1IndividualMOOP,0.0000,6350.0000,657.5349
1,MEHBInnTier1FamilyMOOP,0.0000,12700.0000,1486.2910
2,MEHBOutOfNetIndividualMOOP,0.0000,18000.0000,1192.1041
3,MEHBDedInnTier1Individual,0.0000,6850.0000,1389.5324
4,MEHBDedInnTier1Family,0.0000,13200.0000,3314.0736
5,EHBPercentTotalPremium,0.8835,1.0000,0.9948


### Nulos nas colunas financeiras chave

,column,null_count,null_pct
4,MEHBInnTier1IndividualMOOP,65734,84.9800
6,EHBPercentTotalPremium,54243,70.1200
5,MEHBDedInnTier1Individual,42833,55.3700
8,WellnessProgramOffered,11660,15.0700
3,ServiceAreaId,15,0.0200
0,MetalLevel,0,0.0000
2,NetworkId,0,0.0000
1,PlanType,0,0.0000
7,IsNewPlan,0,0.0000


In [16]:
display(Markdown("### Variantes CSR — distribuição de CSRVariationType"))
display(categorical_dist("plan_attributes", "CSRVariationType"))

### Variantes CSR — distribuição de CSRVariationType

,value,n,pct
0,Zero Cost Sharing Plan Variation,10745,13.8900
1,Limited Cost Sharing Plan Variation,10745,13.8900
2,Standard Silver On Exchange Plan,6054,7.8300
3,Standard Silver Off Exchange Plan,5303,6.8600
4,Standard Bronze On Exchange Plan,4553,5.8900
5,Standard Gold On Exchange Plan,4494,5.8100
6,Standard High Off Exchange Plan,4098,5.3000
7,94% AV Level Silver Plan,4071,5.2600
8,87% AV Level Silver Plan,4071,5.2600
9,73% AV Level Silver Plan,4071,5.2600


In [17]:
display(Markdown("""
### Notas para Gold

- **Tabela dimensional central** — praticamente todas as métricas da Gold passam por join com `plan_attributes`.
- **Filtro CSR recomendado:** excluir variantes CSR (`CSRVariationType != '"Standard Platinum Plan"'`
  ou filtrar pelo sufixo do `PlanId`) para evitar duplicatas nas análises de preço.
  Os planos base têm `VariantId` terminando em `00`.
- **Q3 (benefícios × preço):** fazer join `plan_attributes` → `benefits_cost_sharing` → `rate`
  para correlacionar riqueza de benefícios com `IndividualRate`.
- **MOOP como indicador de exposição financeira:** `MEHBInnTier1IndividualMOOP` é o teto máximo
  que o paciente paga por ano — relevante para Q1 (oncologia).
- **178 colunas:** para Gold, selecionar apenas as necessárias por análise;
  não trazer a tabela inteira nos joins.
"""))


### Notas para Gold

- **Tabela dimensional central** — praticamente todas as métricas da Gold passam por join com `plan_attributes`.
- **Filtro CSR recomendado:** excluir variantes CSR (`CSRVariationType != '"Standard Platinum Plan"'`
  ou filtrar pelo sufixo do `PlanId`) para evitar duplicatas nas análises de preço.
  Os planos base têm `VariantId` terminando em `00`.
- **Q3 (benefícios × preço):** fazer join `plan_attributes` → `benefits_cost_sharing` → `rate`
  para correlacionar riqueza de benefícios com `IndividualRate`.
- **MOOP como indicador de exposição financeira:** `MEHBInnTier1IndividualMOOP` é o teto máximo
  que o paciente paga por ano — relevante para Q1 (oncologia).
- **178 colunas:** para Gold, selecionar apenas as necessárias por análise;
  não trazer a tabela inteira nos joins.


---
## 6. Tabela: `benefits_cost_sharing`

**Descrição:** Benefícios cobertos por cada plano e regras de custo (copay/coinsurance). Uma linha por benefício por plano.

**Dimensão:** ~5 milhões de linhas · 34 colunas

**Transformações Silver aplicadas:**
- `CopayInnTier1/2`, `CoinsInnTier1/2`, `CopayOutofNet`, `CoinsOutofNet` → `double`
  (tratando `"$30"` → 30.0, `"No Charge"` → 0.0, `"Not Applicable"` → null)
- `IsCovered`, `IsEHB`, `QuantLimitOnSvc`, `IsSubjToDedTier1/2` → `boolean`
- `LimitQty`, `BusinessYear` → `integer`
- Deduplicação por `BenefitName + BusinessYear + PlanId` mantendo maior `VersionNum`

**Colunas relevantes para Gold:**
- `PlanId` — join com `plan_attributes` (MetalLevel) e `rate` (preço)
- `BenefitName` — filtro para análises específicas (ex: `'Chemotherapy'`)
- `IsCovered` — se o plano cobre o benefício
- `CopayInnTier1` — copagamento fixo em rede (tipo $30)
- `CoinsInnTier1` — coassegurado em % (tipo 0.20 = 20%)
- `IsSubjToDedTier1` — se o custo é sujeito ao deductible primeiro
- `BusinessYear` — análise temporal 2014-2016

In [18]:
display(Markdown("### Schema"))
display(get_schema("benefits_cost_sharing"))

### Schema

,Column Name,Type,Partition
0,benefitname,string,False
1,businessyear,int,False
2,coinsinntier1,double,False
3,coinsinntier2,double,False
4,coinsoutofnet,double,False
5,copayinntier1,double,False
6,copayinntier2,double,False
7,copayoutofnet,double,False
8,ehbvarreason,string,False
9,exclusions,string,False


In [19]:
display(Markdown("### Amostra — benefícios oncológicos (Chemotherapy / Radiation Therapy)"))
display(query("""
    SELECT
        BusinessYear, StateCode, PlanId, BenefitName,
        IsCovered, IsEHB, IsSubjToDedTier1,
        CopayInnTier1, CoinsInnTier1,
        CopayInnTier2, CoinsInnTier2,
        CopayOutofNet, CoinsOutofNet,
        LimitQty, LimitUnit
    FROM "eedb015_silver"."benefits_cost_sharing"
    WHERE BenefitName IN ('Chemotherapy', 'Radiation Therapy', 'Infusion Therapy')
    LIMIT 15
"""))

### Amostra — benefícios oncológicos (Chemotherapy / Radiation Therapy)

,BusinessYear,StateCode,PlanId,BenefitName,IsCovered,IsEHB,IsSubjToDedTier1,CopayInnTier1,CoinsInnTier1,CopayInnTier2,CoinsInnTier2,CopayOutofNet,CoinsOutofNet,LimitQty,LimitUnit
0,2014,NJ,10191NJ0030001-01,Chemotherapy,True,True,True,NaN,NaN,NaN,NaN,0.0000,1.0000,<NA>,<NA>
1,2014,NJ,10191NJ0030001-02,Chemotherapy,True,True,True,0.0000,0.0000,NaN,NaN,0.0000,0.0000,<NA>,<NA>
2,2014,NJ,10191NJ0030001-03,Chemotherapy,True,True,True,NaN,NaN,NaN,NaN,0.0000,1.0000,<NA>,<NA>
3,2014,NJ,10191NJ0030002-06,Chemotherapy,True,True,True,NaN,NaN,NaN,NaN,0.0000,1.0000,<NA>,<NA>
4,2014,NJ,10191NJ0040001-01,Chemotherapy,True,True,True,NaN,NaN,NaN,NaN,0.0000,1.0000,<NA>,<NA>
5,2014,NJ,10191NJ0040001-02,Chemotherapy,True,True,True,0.0000,0.0000,NaN,NaN,0.0000,0.0000,<NA>,<NA>
6,2014,NJ,10191NJ0040002-01,Chemotherapy,True,True,True,NaN,NaN,NaN,NaN,0.0000,1.0000,<NA>,<NA>
7,2014,NJ,10191NJ0050001-01,Chemotherapy,True,True,False,20.0000,0.0000,NaN,NaN,0.0000,1.0000,<NA>,<NA>
8,2014,NJ,10191NJ0050001-04,Chemotherapy,True,True,False,20.0000,0.0000,NaN,NaN,0.0000,1.0000,<NA>,<NA>
9,2014,NJ,10191NJ0050002-00,Chemotherapy,True,True,False,10.0000,0.0000,NaN,NaN,0.0000,1.0000,<NA>,<NA>


In [20]:
display(Markdown("### Top 30 BenefitName mais frequentes"))
display(categorical_dist("benefits_cost_sharing", "BenefitName", limit=30))

### Top 30 BenefitName mais frequentes

,value,n,pct
0,Routine Dental Services (Adult),77323,1.5300
1,Orthodontia - Adult,77323,1.5300
2,Major Dental Care - Child,77323,1.5300
3,Major Dental Care - Adult,77323,1.5300
4,Dental Check-Up for Children,77323,1.5300
5,Basic Dental Care - Adult,77323,1.5300
6,Basic Dental Care - Child,77323,1.5300
7,Orthodontia - Child,77315,1.5300
8,Accidental Dental,76914,1.5300
9,Treatment for Temporomandibular Joint Disorders,65694,1.3000


In [21]:
display(Markdown("### Perfil de Nulos — colunas de custo"))
display(null_profile("benefits_cost_sharing", [
    "CopayInnTier1", "CoinsInnTier1",
    "CopayInnTier2", "CoinsInnTier2",
    "CopayOutofNet", "CoinsOutofNet",
    "IsCovered", "IsEHB", "IsSubjToDedTier1", "LimitQty",
]))

### Perfil de Nulos — colunas de custo

,column,null_count,null_pct
3,CoinsInnTier2,4883114,96.8300
2,CopayInnTier2,4818496,95.5400
9,LimitQty,4355835,86.3700
1,CoinsInnTier1,3612024,71.6200
5,CoinsOutofNet,2648728,52.5200
4,CopayOutofNet,2496378,49.5000
8,IsSubjToDedTier1,2464220,48.8600
0,CopayInnTier1,2350573,46.6100
7,IsEHB,1816367,36.0200
6,IsCovered,215972,4.2800


In [22]:
display(Markdown("### Análise Oncológica — Copay × Coinsurance por Ano"))
display(query("""
    SELECT
        BusinessYear,
        BenefitName,
        COUNT(*)                                                AS total_plans,
        SUM(CASE WHEN IsCovered = TRUE THEN 1 ELSE 0 END)      AS covered_plans,
        SUM(CASE WHEN CopayInnTier1 = 0 THEN 1 ELSE 0 END)    AS copay_zero_plans,
        SUM(CASE WHEN CoinsInnTier1 = 0 THEN 1 ELSE 0 END)    AS coins_zero_plans,
        ROUND(AVG(CopayInnTier1), 2)                           AS avg_copay,
        ROUND(AVG(CoinsInnTier1), 4)                           AS avg_coinsurance
    FROM "eedb015_silver"."benefits_cost_sharing"
    WHERE BenefitName IN ('Chemotherapy', 'Radiation Therapy', 'Infusion Therapy')
    GROUP BY BusinessYear, BenefitName
    ORDER BY BenefitName, BusinessYear
"""))

### Análise Oncológica — Copay × Coinsurance por Ano

,BusinessYear,BenefitName,total_plans,covered_plans,copay_zero_plans,coins_zero_plans,avg_copay,avg_coinsurance
0,2014,Chemotherapy,15157,14891,12861,3424,2.9400,0.0399
1,2015,Chemotherapy,26991,26738,24040,5581,1.5200,0.0243
2,2016,Chemotherapy,23482,23304,4077,4191,6.4800,0.0231
3,2014,Infusion Therapy,15157,14606,12499,3447,3.4100,0.0411
4,2015,Infusion Therapy,26991,26157,23367,5502,1.8400,0.0286
5,2016,Infusion Therapy,23482,23339,3855,3913,10.6100,0.0285


In [23]:
display(Markdown("""
### Notas para Gold

- **Q1 (Oncologia):** filtrar `BenefitName IN ('Chemotherapy', 'Radiation Therapy', 'Infusion Therapy')`,
  depois join com `plan_attributes` para trazer `MetalLevel`. Análise temporal de `CopayInnTier1`
  e `CoinsInnTier1` por ano e nível metálico.
- **Q3 (Benefícios × Preço):** agregar score de cobertura por plano (ex: % de EHBs cobertos,
  média de copay), depois join com `rate` para correlacionar com `IndividualRate`.
- **Null semantics:** `CoinsInnTier1 IS NULL` significa "não aplicável" (copay fixo);
  `CopayInnTier1 IS NULL` significa "coinsurance aplicado". Na Gold, criar coluna `cost_type`
  = 'copay' / 'coinsurance' / 'none' para facilitar análises.
- **IsSubjToDedTier1 = TRUE:** o custo só é aplicado depois de atingir o deductible — impacta
  significativamente a exposição real do paciente crônico (Q1).
"""))


### Notas para Gold

- **Q1 (Oncologia):** filtrar `BenefitName IN ('Chemotherapy', 'Radiation Therapy', 'Infusion Therapy')`,
  depois join com `plan_attributes` para trazer `MetalLevel`. Análise temporal de `CopayInnTier1`
  e `CoinsInnTier1` por ano e nível metálico.
- **Q3 (Benefícios × Preço):** agregar score de cobertura por plano (ex: % de EHBs cobertos,
  média de copay), depois join com `rate` para correlacionar com `IndividualRate`.
- **Null semantics:** `CoinsInnTier1 IS NULL` significa "não aplicável" (copay fixo);
  `CopayInnTier1 IS NULL` significa "coinsurance aplicado". Na Gold, criar coluna `cost_type`
  = 'copay' / 'coinsurance' / 'none' para facilitar análises.
- **IsSubjToDedTier1 = TRUE:** o custo só é aplicado depois de atingir o deductible — impacta
  significativamente a exposição real do paciente crônico (Q1).


---
## 7. Tabela: `service_area`

**Descrição:** Áreas geográficas (estado, condado, ZIP) onde cada seguradora opera.

**Dimensão:** ~42.000 linhas · 20 colunas

**Transformações Silver aplicadas:**
- `CoverEntireState`, `PartialCounty` → `boolean`
- `BusinessYear` → `integer`

**Colunas relevantes para Gold:**
- `ServiceAreaId` — join com `plan_attributes`
- `IssuerId` — identificador da seguradora
- `StateCode` — dimensão geográfica
- `CoverEntireState` — se cobre o estado inteiro
- `County` + `CountyCode` — granularidade para análises de deserto de cobertura
- `BusinessYear` — dimensão temporal

In [24]:
display(Markdown("### Schema"))
display(get_schema("service_area"))

### Schema

,Column Name,Type,Partition
0,businessyear,int,False
1,statecode,string,False
2,issuerid,string,False
3,sourcename,string,False
4,versionnum,string,False
5,importdate,timestamp,False
6,issuerid2,string,False
7,statecode2,string,False
8,serviceareaid,string,False
9,serviceareaname,string,False


In [25]:
display(Markdown("### Amostra (10 linhas)"))
display(sample_table("service_area"))

### Amostra (10 linhas)

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,statecode2,serviceareaid,serviceareaname,coverentirestate,county,partialcounty,zipcodes,partialcountyjustification,rownumber,marketcoverage,dentalonlyplan,landinzone_path,ingestion_datetime
0,2014,PA,22444,HIOS,9,NaT,22444,PA,PAS001,Geisinger Health Plan,False,42103.0,False,<NA>,<NA>,42,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
1,2014,PA,22444,HIOS,9,NaT,22444,PA,PAS001,Geisinger Health Plan,False,42105.0,False,<NA>,<NA>,43,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
2,2014,PA,22444,HIOS,9,NaT,22444,PA,PAS001,Geisinger Health Plan,False,42107.0,False,<NA>,<NA>,44,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
3,2014,PA,22444,HIOS,9,NaT,22444,PA,PAS001,Geisinger Health Plan,False,42109.0,False,<NA>,<NA>,45,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
4,2014,PA,22444,HIOS,9,NaT,22444,PA,PAS001,Geisinger Health Plan,False,42111.0,False,<NA>,<NA>,46,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
5,2014,PA,22444,HIOS,9,NaT,22444,PA,PAS001,Geisinger Health Plan,False,42113.0,False,<NA>,<NA>,47,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
6,2014,PA,22444,HIOS,9,NaT,22444,PA,PAS001,Geisinger Health Plan,False,42115.0,False,<NA>,<NA>,48,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
7,2014,PA,22444,HIOS,9,NaT,22444,PA,PAS001,Geisinger Health Plan,False,42117.0,False,<NA>,<NA>,49,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
8,2014,PA,22444,HIOS,9,NaT,22444,PA,PAS001,Geisinger Health Plan,False,42119.0,False,<NA>,<NA>,50,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
9,2014,PA,22444,HIOS,9,NaT,22444,PA,PAS001,Geisinger Health Plan,False,42127.0,False,<NA>,<NA>,51,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00


In [26]:
display(Markdown("### Distribuição: CoverEntireState por Ano"))
display(query("""
    SELECT
        BusinessYear,
        CoverEntireState,
        COUNT(*)                  AS n,
        COUNT(DISTINCT IssuerId)  AS distinct_issuers,
        COUNT(DISTINCT StateCode) AS distinct_states
    FROM "eedb015_silver"."service_area"
    GROUP BY BusinessYear, CoverEntireState
    ORDER BY BusinessYear, CoverEntireState
"""))

display(Markdown("### Nulos — campos geográficos"))
display(null_profile("service_area", [
    "County", "ZipCodes", "PartialCounty",
    "ServiceAreaId", "ServiceAreaName",
]))

### Distribuição: CoverEntireState por Ano

,BusinessYear,CoverEntireState,n,distinct_issuers,distinct_states
0,2014,False,8473,185,31
1,2014,True,401,298,36
2,2015,False,16825,303,36
3,2015,True,670,531,37
4,2016,False,15154,299,37
5,2016,True,724,541,38


### Nulos — campos geográficos

,column,null_count,null_pct
1,ZipCodes,41631,98.5400
0,County,1795,4.2500
2,PartialCounty,1795,4.2500
3,ServiceAreaId,0,0.0000
4,ServiceAreaName,0,0.0000


In [27]:
display(Markdown("### Competição por Estado — número de seguradoras por estado/ano"))
display(query("""
    SELECT
        BusinessYear,
        StateCode,
        COUNT(DISTINCT IssuerId) AS num_issuers
    FROM "eedb015_silver"."service_area"
    GROUP BY BusinessYear, StateCode
    ORDER BY BusinessYear, num_issuers DESC
    LIMIT 30
"""))

### Competição por Estado — número de seguradoras por estado/ano

,BusinessYear,StateCode,num_issuers
0,2014,TX,27
1,2014,MI,25
2,2014,PA,25
3,2014,FL,24
4,2014,WI,22
5,2014,AZ,21
6,2014,OH,21
7,2014,VA,18
8,2014,IL,17
9,2014,GA,16


In [28]:
display(Markdown("""
### Notas para Gold

- **Q2 / Q5 (Competição e monopólios):** a query acima de `COUNT(DISTINCT IssuerId) por StateCode`
  é a base para a tabela Gold de competição. Combinar com preço médio de `rate`.
- **Q5 (Desertos de cobertura):** filtrar `CoverEntireState = FALSE` e analisar `County` +
  `CountyCode` para identificar condados cobertos por apenas 1 seguradora.
- **Join com plan_attributes:** `service_area.ServiceAreaId = plan_attributes.ServiceAreaId`
  permite ligar a cobertura geográfica a MetalLevel e PlanType.
- **Atenção:** `ZipCodes` tem alta taxa de nulos (cobertura por condado é o padrão;
  ZIP é granularidade excepcional).
"""))


### Notas para Gold

- **Q2 / Q5 (Competição e monopólios):** a query acima de `COUNT(DISTINCT IssuerId) por StateCode`
  é a base para a tabela Gold de competição. Combinar com preço médio de `rate`.
- **Q5 (Desertos de cobertura):** filtrar `CoverEntireState = FALSE` e analisar `County` +
  `CountyCode` para identificar condados cobertos por apenas 1 seguradora.
- **Join com plan_attributes:** `service_area.ServiceAreaId = plan_attributes.ServiceAreaId`
  permite ligar a cobertura geográfica a MetalLevel e PlanType.
- **Atenção:** `ZipCodes` tem alta taxa de nulos (cobertura por condado é o padrão;
  ZIP é granularidade excepcional).


---
## 8. Tabela: `business_rules`

**Descrição:** Regras operacionais e de elegibilidade por plano (idade mínima/máxima, carência, programas de saúde).

**Dimensão:** ~21.000 linhas · 25 colunas

**Transformações Silver aplicadas:**
- `DentalOnlyPlan`, `WellnessProgramOffered` → `boolean`
- `BusinessYear`, `BeginPrimaryCareCostSharingAfterNumberOfVisits`, `InpatientCopaymentMaximumDays` → `integer`
- `FirstTierUtilization`, `SecondTierUtilization` → `double`

**Colunas relevantes para Gold:**
- `PlanId` — join com `plan_attributes`
- `MarketCoverage` — Individual vs. Small Group
- `DentalOnlyPlan` — filtro para excluir planos odontológicos das análises
- `BeginPrimaryCareCostSharingAfterNumberOfVisits` — visitas gratuitas antes de custo
- `WellnessProgramOffered` — feature para análise de benefícios × preço

In [29]:
display(Markdown("### Schema"))
display(get_schema("business_rules"))

### Schema

,Column Name,Type,Partition
0,businessyear,int,False
1,statecode,string,False
2,issuerid,string,False
3,sourcename,string,False
4,versionnum,string,False
5,importdate,timestamp,False
6,issuerid2,string,False
7,tin,string,False
8,productid,string,False
9,standardcomponentid,string,False


In [30]:
display(Markdown("### Amostra (10 linhas)"))
display(sample_table("business_rules"))

### Amostra (10 linhas)

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,tin,productid,standardcomponentid,enrolleecontractratedeterminationrule,twoparentfamilymaxdependentsrule,singleparentfamilymaxdependentsrule,dependentmaximumagrule,childrenonlycontractmaxchildrenrule,domesticpartnerasspouseindicator,samesexpartnerasspouseindicator,agedeterminationrule,minimumtobaccofreemonthsrule,cohabitationrule,rownumber,marketcoverage,dentalonlyplan,landinzone_path,ingestion_datetime
0,2014,AL,82285,HIOS,7,NaT,82285,94-2761537,<NA>,82285AL0010006,A different rate (specifically for parties of two or more)for each enrollee ...,3 or more,3 or more,26,3 or more,True,True,Age on effective date,<NA>,"Spouse,No;Adopted Child,No;Foster Child,No;Ward,No;Stepson or Stepdaughter,N...",14,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
1,2014,AL,82285,HIOS,7,NaT,82285,94-2761537,<NA>,82285AL0020001,A different rate (specifically for parties of two or more)for each enrollee ...,3 or more,3 or more,<NA>,3 or more,True,True,Age on effective date,<NA>,"Adopted Child,Yes;Foster Child,Yes;Stepson or Stepdaughter,Yes;Child,Yes",15,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
2,2014,AL,82285,HIOS,7,NaT,82285,94-2761537,<NA>,82285AL0020002,A different rate (specifically for parties of two or more)for each enrollee ...,3 or more,3 or more,<NA>,3 or more,True,True,Age on effective date,<NA>,"Adopted Child,Yes;Foster Child,Yes;Stepson or Stepdaughter,Yes;Child,Yes",16,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
3,2014,AL,82285,HIOS,7,NaT,82285,94-2761537,<NA>,82285AL0020004,A different rate (specifically for parties of two or more)for each enrollee ...,3 or more,3 or more,26,3 or more,True,True,Age on effective date,<NA>,"Spouse,No;Adopted Child,No;Foster Child,No;Ward,No;Stepson or Stepdaughter,N...",17,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
4,2014,AL,82285,HIOS,7,NaT,82285,94-2761537,<NA>,82285AL0020006,A different rate (specifically for parties of two or more)for each enrollee ...,3 or more,3 or more,26,3 or more,True,True,Age on effective date,<NA>,"Spouse,No;Adopted Child,No;Foster Child,No;Ward,No;Stepson or Stepdaughter,N...",18,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
5,2014,AZ,17100,HIOS,7,NaT,17100,13-5581829,<NA>,<NA>,There are rates specifically for couples and for families (not just addition...,3 or more,3 or more,26,3 or more,True,True,Age on effective date,<NA>,"Spouse,No;Grandson or Granddaughter,Yes;Adopted Child,Yes;Foster Child,Yes;W...",10,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
6,2014,AZ,17100,HIOS,7,NaT,17100,13-5581829,<NA>,17100AZ0160001,A different rate (specifically for parties of two or more)for each enrollee ...,3 or more,3 or more,21,3 or more,False,False,Age on effective date,<NA>,"Brother or Sister,Yes;Ward,Yes;Self,Yes;Child,Yes;Dependent on a Minor Depen...",11,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
7,2014,AZ,18156,HIOS,1,NaT,18156,35-0472300,<NA>,<NA>,There are rates specifically for couples and for families (not just addition...,3 or more,3 or more,30,1,True,True,Age on effective date,<NA>,"Spouse,No;Grandson or Granddaughter,Yes;Adopted Child,No;Foster Child,Yes;St...",10,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
8,2014,AZ,23307,HIOS,8,NaT,23307,61-1013183,<NA>,<NA>,A different rate (specifically for parties of two or more)for each enrollee ...,3 or more,3 or more,26,3 or more,True,False,Age on effective date,6,"Spouse,No;Adopted Child,No;Ste

In [31]:
display(Markdown("### Distribuição: MarketCoverage"))
display(categorical_dist("business_rules", "MarketCoverage"))

display(Markdown("### Nulos e Estatísticas — colunas de regras"))
display(null_profile("business_rules", [
    "MarketCoverage", "DentalOnlyPlan",
    # "BeginPrimaryCareCostSharingAfterNumberOfVisits",
    # "BeginPrimaryCareDeductibleCoinsuranceAfterNumberOfCopays",
    # "InpatientCopaymentMaximumDays",
    # "FirstTierUtilization", "SecondTierUtilization",
]))

### Distribuição: MarketCoverage

,value,n,pct
0,Individual,11043,52.3700
1,SHOP (Small Group),8834,41.9000
2,<NA>,1208,5.7300


### Nulos e Estatísticas — colunas de regras

,column,null_count,null_pct
0,MarketCoverage,1208,5.7300
1,DentalOnlyPlan,1208,5.7300


In [32]:
display(Markdown("""
### Notas para Gold

- **Filtro obrigatório na maioria das análises:** `WHERE DentalOnlyPlan = FALSE`
  para excluir planos odontológicos que distorcem métricas de prêmio e benefícios médicos.
- **MarketCoverage:** filtrar `MarketCoverage = 'Individual'` para as análises
  das questões do projeto (mercado individual, não SHOP/small group).
- **Q3 (Benefícios × Preço):** `WellnessProgramOffered` é uma feature candidata
  para o modelo de precificação.
- Tabela relativamente pequena — pode ser broadcast join no Spark Gold.
"""))


### Notas para Gold

- **Filtro obrigatório na maioria das análises:** `WHERE DentalOnlyPlan = FALSE`
  para excluir planos odontológicos que distorcem métricas de prêmio e benefícios médicos.
- **MarketCoverage:** filtrar `MarketCoverage = 'Individual'` para as análises
  das questões do projeto (mercado individual, não SHOP/small group).
- **Q3 (Benefícios × Preço):** `WellnessProgramOffered` é uma feature candidata
  para o modelo de precificação.
- Tabela relativamente pequena — pode ser broadcast join no Spark Gold.


---
## 9. Tabelas: `crosswalk2015` e `crosswalk2016`

**Descrição:** Mapeamento de linhagem temporal dos planos — liga o `PlanId` de um ano ao equivalente no ano seguinte.

**Dimensões:**
- `crosswalk2015`: ~132.000 linhas · 23 colunas (2014 → 2015)
- `crosswalk2016`: ~150.000 linhas · 23 colunas (2015 → 2016)

**Transformações Silver aplicadas:**
- `CrosswalkLevel` (inteiro), `BusinessYear` → `integer`

**Colunas relevantes para Gold:**
- `CrosswalkPlanId` — PlanId no ano de origem
- `PlanId` — PlanId no ano de destino
- `CrosswalkLevel` — tipo de transição (ver distribuição abaixo)
- `StateCode`, `IssuerId` — filtragem

In [33]:
display(Markdown("### Schema crosswalk2015"))
display(get_schema("crosswalk2015"))

### Schema crosswalk2015

,Column Name,Type,Partition
0,state,string,False
1,dentalplan,boolean,False
2,planid_2014,string,False
3,issuerid_2014,string,False
4,multistateplan_2014,boolean,False
5,metallevel_2014,string,False
6,childadultonly_2014,int,False
7,fipscode,string,False
8,zipcode,string,False
9,crosswalklevel,int,False


In [34]:
display(Markdown("### Amostra — crosswalk2015 (10 linhas)"))
display(query("""
    SELECT State, issuerid_2014, issuerid_2015, planid_2014, planid_2015, CrosswalkLevel
    FROM "eedb015_silver"."crosswalk2015"
    LIMIT 10
"""))

display(Markdown("### Amostra — crosswalk2016 (10 linhas)"))
display(query("""
    SELECT State, issuerid_2015, issuerid_2016, planid_2015, planid_2016, CrosswalkLevel
    FROM "eedb015_silver"."crosswalk2016"
    LIMIT 10
"""))

### Amostra — crosswalk2015 (10 linhas)

,State,issuerid_2014,issuerid_2015,planid_2014,planid_2015,CrosswalkLevel
0,AK,21989,21989,21989AK0010001,21989AK0030001,1
1,AK,21989,21989,21989AK0010001,21989AK0030001,1
2,AK,21989,21989,21989AK0010001,21989AK0030001,1
3,AK,21989,21989,21989AK0010001,21989AK0030001,1
4,AK,21989,21989,21989AK0010001,21989AK0030001,1
5,AK,21989,21989,21989AK0010001,21989AK0030001,1
6,AK,21989,21989,21989AK0010001,21989AK0030001,1
7,AK,21989,21989,21989AK0010001,21989AK0030001,1
8,AK,21989,21989,21989AK0010001,21989AK0030001,1
9,AK,21989,21989,21989AK0010001,21989AK0030001,1


### Amostra — crosswalk2016 (10 linhas)

,State,issuerid_2015,issuerid_2016,planid_2015,planid_2016,CrosswalkLevel
0,OR,10091,10091,10091OR0360004,10091OR0360004,0
1,OR,10091,10091,10091OR0360004,10091OR0360004,0
2,OR,10091,10091,10091OR0360004,10091OR0360004,0
3,OR,10091,10091,10091OR0360004,10091OR0360004,0
4,OR,10091,10091,10091OR0360004,10091OR0360004,0
5,OR,10091,10091,10091OR0360004,10091OR0360004,0
6,OR,10091,10091,10091OR0360004,10091OR0360004,0
7,OR,10091,10091,10091OR0360004,10091OR0360004,0
8,OR,10091,10091,10091OR0360004,10091OR0360004,0
9,OR,10091,10091,10091OR0360004,10091OR0360004,0


In [36]:
display(Markdown("### Distribuição: CrosswalkLevel (crosswalk2015)"))
display(categorical_dist("crosswalk2015", "CrosswalkLevel"))

display(Markdown("### Distribuição: CrosswalkLevel (crosswalk2016)"))
display(categorical_dist("crosswalk2016", "CrosswalkLevel"))

display(Markdown("### Nulos — chave de destino PlanId"))
display(null_profile("crosswalk2015", ["planid_2014", "planid_2015"]))
display(null_profile("crosswalk2016", ["planid_2015", "planid_2016"]))

### Distribuição: CrosswalkLevel (crosswalk2015)

,value,n,pct
0,0,70756,53.4000
1,1,31438,23.7300
2,2,17177,12.9600
3,4,7661,5.7800
4,3,5473,4.1300


### Distribuição: CrosswalkLevel (crosswalk2016)

,value,n,pct
0,0,82294,54.8600
1,1,34248,22.8300
2,2,15301,10.2000
3,4,14552,9.7000
4,3,2245,1.5000
5,5,1365,0.9100


### Nulos — chave de destino PlanId

,column,null_count,null_pct
0,planid_2014,0,0.0000
1,planid_2015,0,0.0000


,column,null_count,null_pct
0,planid_2015,0,0.0000
1,planid_2016,0,0.0000


In [38]:
display(Markdown("### Taxa de continuidade dos planos 2014 → 2015 → 2016"))
display(query("""
    SELECT
        c15.state,
        COUNT(DISTINCT c15.planid_2014)   AS plans_2014,
        COUNT(DISTINCT c15.planid_2015)   AS plans_2015_mapped,
        COUNT(DISTINCT c16.planid_2016)   AS plans_2016_mapped
    FROM "eedb015_silver"."crosswalk2015" c15
    LEFT JOIN "eedb015_silver"."crosswalk2016" c16
        ON c15.planid_2015 = c16.planid_2015
    GROUP BY c15.state
    ORDER BY plans_2014 DESC
    LIMIT 20
"""))

### Taxa de continuidade dos planos 2014 → 2015 → 2016

,state,plans_2014,plans_2015_mapped,plans_2016_mapped
0,FL,276,157,118
1,WI,268,263,202
2,OH,226,170,145
3,AZ,223,118,61
4,PA,217,156,123
5,TX,169,309,391
6,MI,154,113,70
7,IL,135,133,171
8,VA,133,91,85
9,IN,128,56,37


In [39]:
display(Markdown("""
### Notas para Gold

- **Q1 (Evolução temporal oncológica):** construir tabela Gold `plan_lineage` que une os três anos
  via dois joins sequenciais: `crosswalk2015.CrosswalkPlanId` (2014) → `crosswalk2015.PlanId` (2015)
  → `crosswalk2016.PlanId` (2016).
- **Nulos em `PlanId` de destino** = planos descontinuados. Tratar como saída do mercado na análise
  de linhagem.
- **CrosswalkLevel** é inteiro na Silver (foi convertido de string para int). Verificar o
  mapeamento de valores na documentação CMS para rotular os tipos de transição.
- **Duplicatas:** verificar se há casos onde 1 `CrosswalkPlanId` mapeia para múltiplos `PlanId`
  (fusões/splits de planos) — podem gerar fan-out em joins na Gold.
"""))


### Notas para Gold

- **Q1 (Evolução temporal oncológica):** construir tabela Gold `plan_lineage` que une os três anos
  via dois joins sequenciais: `crosswalk2015.CrosswalkPlanId` (2014) → `crosswalk2015.PlanId` (2015)
  → `crosswalk2016.PlanId` (2016).
- **Nulos em `PlanId` de destino** = planos descontinuados. Tratar como saída do mercado na análise
  de linhagem.
- **CrosswalkLevel** é inteiro na Silver (foi convertido de string para int). Verificar o
  mapeamento de valores na documentação CMS para rotular os tipos de transição.
- **Duplicatas:** verificar se há casos onde 1 `CrosswalkPlanId` mapeia para múltiplos `PlanId`
  (fusões/splits de planos) — podem gerar fan-out em joins na Gold.


---
## 10. Tabela: `network`

**Descrição:** Metadados das redes de prestadores de cada seguradora.

**Dimensão:** ~3.800 linhas · 16 colunas

**Transformações Silver aplicadas:**
- `BusinessYear` → `integer`
- Sem outras transformações relevantes (tabela majoritariamente textual)

**Colunas relevantes para Gold:**
- `NetworkId` — join com `plan_attributes`
- `IssuerId` — seguradora
- `NetworkName` — nome da rede (proxy qualitativo do tipo de rede)
- `StateCode` — estado
- `BusinessYear` — temporal

In [40]:
display(Markdown("### Schema"))
display(get_schema("network"))

### Schema

,Column Name,Type,Partition
0,businessyear,int,False
1,statecode,string,False
2,issuerid,string,False
3,sourcename,string,False
4,versionnum,string,False
5,importdate,timestamp,False
6,issuerid2,string,False
7,statecode2,string,False
8,networkname,string,False
9,networkid,string,False


In [41]:
display(Markdown("### Amostra (10 linhas)"))
display(sample_table("network"))

### Amostra (10 linhas)

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,statecode2,networkname,networkid,networkurl,rownumber,marketcoverage,dentalonlyplan,landinzone_path,ingestion_datetime
0,2014,AK,21989,HIOS,6,NaT,21989,AK,ODS Premier,AKN001,https://www.modahealth.com/ProviderSearch/faces/webpages/home.xhtml?dn=ods,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
1,2014,AK,38344,HIOS,6,NaT,38344,AK,HeritagePlus,AKN001,https://www.premera.com/wa/visitor/,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
2,2014,AK,38536,HIOS,2,NaT,38536,AK,Lincoln Dental Connect,AKN001,http://lfg.go2dental.com/member/dental_search/searchprov.cgi?P=LFGDentalConn...,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
3,2014,AK,42507,HIOS,3,NaT,42507,AK,DentalGuard Preferred,AKN001,https://www.guardiananytime.com/fpapp/FPWeb/dentalSearch.process,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
4,2014,AK,73836,HIOS,6,NaT,73836,AK,Moda Plus AK Regional,AKN001,https://www.modahealth.com/ProviderSearch/faces/webpages/home.xhtml?dn=ods,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
5,2014,AK,73836,HIOS,6,NaT,73836,AK,Moda Plus Providence,AKN002,https://www.modahealth.com/ProviderSearch/faces/webpages/home.xhtml?dn=ods,14,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
6,2014,AK,74819,HIOS,7,NaT,74819,AK,DenteMax,AKN001,http://www2.dentemax.com/,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
7,2014,AK,84859,HIOS,1,NaT,84859,AK,Ameritas PPO Dental Network,AKN001,www.standard.com,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
8,2014,AL,12538,HIOS,8,NaT,12538,AL,DenteMax,ALN001,http://www2.dentemax.com/,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
9,2014,AL,12538,HIOS,8,NaT,12538,AL,CONNECTION Dental through PPO USA,ALN002,http://www.ppousa.com/patients/index.html,14,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00


In [42]:
display(Markdown("### Dimensionamento por Ano"))
display(query("""
    SELECT
        BusinessYear,
        COUNT(*)                   AS total_rows,
        COUNT(DISTINCT NetworkId)  AS distinct_networks,
        COUNT(DISTINCT IssuerId)   AS distinct_issuers,
        COUNT(DISTINCT StateCode)  AS distinct_states
    FROM "eedb015_silver"."network"
    GROUP BY BusinessYear
    ORDER BY BusinessYear
"""))

display(Markdown("### Nulos"))
display(null_profile("network", ["NetworkId", "NetworkName", "NetworkURL", "IssuerId", "StateCode"]))

### Dimensionamento por Ano

,BusinessYear,total_rows,distinct_networks,distinct_issuers,distinct_states
0,2014,937,179,444,36
1,2015,1459,248,783,37
2,2016,1426,232,770,38


### Nulos

,column,null_count,null_pct
0,NetworkId,0,0.0000
1,NetworkName,0,0.0000
2,NetworkURL,0,0.0000
3,IssuerId,0,0.0000
4,StateCode,0,0.0000


In [43]:
display(Markdown("### Planos por Rede — quantos planos compartilham a mesma NetworkId"))
display(query("""
    SELECT
        n.BusinessYear,
        n.NetworkId,
        n.NetworkName,
        n.StateCode,
        COUNT(DISTINCT p.PlanId) AS plan_count
    FROM "eedb015_silver"."network" n
    LEFT JOIN "eedb015_silver"."plan_attributes" p
        ON n.NetworkId = p.NetworkId AND n.BusinessYear = p.BusinessYear
    GROUP BY n.BusinessYear, n.NetworkId, n.NetworkName, n.StateCode
    ORDER BY plan_count DESC
    LIMIT 20
"""))

### Planos por Rede — quantos planos compartilham a mesma NetworkId

,BusinessYear,NetworkId,NetworkName,StateCode,plan_count
0,2015,WIN001,DenteMax,WI,2152
1,2015,WIN001,Dental Prime,WI,2152
2,2015,WIN001,Arise Health Plan,WI,2152
3,2015,WIN001,"DenteMax, LLC",WI,2152
4,2015,WIN001,Unity Prime,WI,2152
5,2015,WIN001,EHB Basic Dental Plan (Low),WI,2152
6,2015,WIN001,Dentegra Dental PPO,WI,2152
7,2015,WIN001,Mertier Plus,WI,2152
8,2015,WIN001,Medica Applause,WI,2152
9,2015,WIN001,DenteMaxPlus,WI,2152


In [44]:
display(Markdown("""
### Notas para Gold

- **Q4 (Tamanho da rede × Preço):** proxy de amplitude da rede = `COUNT(DISTINCT PlanId)`
  por `NetworkId` (planos que usam a mesma rede). Redes compartilhadas por mais planos
  tendem a ser maiores.
- **Limitação:** o dataset não contém a lista de prestadores individuais.
  `NetworkName` pode conter dicas qualitativas (ex: 'Narrow', 'HMO', 'Select').
- **Tabela pequena (~3.8K linhas):** pode ser usada como broadcast join no Spark/Gold
  sem preocupação com performance.
- **NetworkURL:** útil apenas para enriquecimento externo (fora do escopo do dataset).
"""))


### Notas para Gold

- **Q4 (Tamanho da rede × Preço):** proxy de amplitude da rede = `COUNT(DISTINCT PlanId)`
  por `NetworkId` (planos que usam a mesma rede). Redes compartilhadas por mais planos
  tendem a ser maiores.
- **Limitação:** o dataset não contém a lista de prestadores individuais.
  `NetworkName` pode conter dicas qualitativas (ex: 'Narrow', 'HMO', 'Select').
- **Tabela pequena (~3.8K linhas):** pode ser usada como broadcast join no Spark/Gold
  sem preocupação com performance.
- **NetworkURL:** útil apenas para enriquecimento externo (fora do escopo do dataset).


---
## 11. Mapa de Joins Silver → Gold

Este diagrama resume os joins necessários para construir cada agregado da Gold.

### Relações entre tabelas Silver

```
network            service_area
  └─ NetworkId         └─ ServiceAreaId
         │                     │
         └────────────────────►plan_attributes◄─────────── business_rules
                                     │  (PlanId)                 (PlanId)
                                     │
                    ┌────────────────┤
                    │                │
                    ▼                ▼
                  rate     benefits_cost_sharing
                (PlanId)          (PlanId)

crosswalk2015.CrosswalkPlanId → crosswalk2015.PlanId
crosswalk2016.CrosswalkPlanId → crosswalk2016.PlanId
```

### Tabelas Gold sugeridas por questão

| Tabela Gold | Questão | Joins necessários | Granularidade |
|---|---|---|---|
| `gold_oncology_copay` | Q1 | `benefits_cost_sharing` + `plan_attributes` (MetalLevel) + `crosswalk` | BenefitName × MetalLevel × BusinessYear |
| `gold_competition_pricing` | Q2 | `rate` + `service_area` (IssuerId count) + `plan_attributes` | StateCode × BusinessYear |
| `gold_benefit_pricing` | Q3 | `benefits_cost_sharing` + `rate` + `plan_attributes` | PlanId × BusinessYear |
| `gold_network_pricing` | Q4 | `network` + `plan_attributes` + `rate` | NetworkId × StateCode × BusinessYear |
| `gold_geographic_monopoly` | Q5 | `service_area` + `rate` + `plan_attributes` | County × StateCode × BusinessYear |

### Joins Silver → Gold: PlanId com sufixo vs. sem sufixo

> **Ver seção "⚠️ Atenção: Formatos de PlanId"** acima para a sintaxe completa.
> Resumo: `rate` e `crosswalk` usam PlanId de 14 chars; `plan_attributes`, `benefits_cost_sharing`
> e `business_rules` usam 17 chars com sufixo `-XX`. Use `SUBSTR(pa.PlanId, 1, 14) = r.PlanId`
> nos joins envolvendo `rate`.

### Filtros comuns recomendados para Gold

```sql
-- Excluir planos odontológicos
WHERE DentalOnlyPlan = FALSE

-- Mercado individual apenas
AND MarketCoverage = 'Individual'

-- Excluir variantes CSR (sufixo PlanId ≠ '00')
AND RIGHT(PlanId, 2) = '00'

-- Excluir outliers de preço
AND IndividualRate > 0 AND IndividualRate < 3000

-- Excluir Age especial
AND Age != 'Family Option'
```